# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and analyze the [FAIR^2](https://sen.science/doi/10.71728/senscience.y7m0-f273) dataset using the `mlcroissant` library and a provided Croissant schema URL. All dataset entities are referenced by their `@id` as required by Croissant best practices.

### Dataset Source

The dataset source is provided via this Croissant schema URL:

```
https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json
```


In [ ]:
# Install mlcroissant if not already present
!pip install -q mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`. This will parse the Croissant schema, fetch file references, and provide programmatic access through the library.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Title: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Published: {metadata.datePublished}")
print(f"License: {metadata.license}")


## 2. Data Overview

Let's examine the record sets, fields, and columns defined in the dataset, referencing each entity by their `@id`. This allows us to programmatically select data for further analysis.

The `mlcroissant` library exposes dataset structure via the `.record_sets` attribute. We will enumerate available record sets and their fields by `@id`.

In [ ]:
# List record sets by @id
print("Record sets available in the dataset:\n")
record_sets = dataset.record_sets
for rs in record_sets:
    print(f"- @id: {rs.id} | name: {rs.name}")
    print("  Fields:")
    for field in rs.fields:
        print(f"   - @id: {field.id} | name: {field.name} | data type: {field.data_type}")
    print()

Now, we can preview records from a selected record set. Replace `<id_of_the_records_set>` below with one of the available record set `@id` values from above. Here, we will use the first listed record set for demonstration.

In [ ]:
# List the first few records from a record set by @id.
if record_sets:
    record_set_id = record_sets[0].id
    print(f"Showing sample records from record_set @id='{record_set_id}':\n")
    for i, record in enumerate(dataset.records(record_set=record_set_id)):
        print(record)
        if i==2:
            break
else:
    print("No record sets found in this dataset.")

## 3. Data Extraction

Let's load data from each available record set, using their `@id`, into pandas DataFrames for exploration. We will store all dataframes in a dictionary indexed by record set `@id`.


In [ ]:
# Build list of all record set @ids
record_set_ids = [rs.id for rs in dataset.record_sets]

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
    else:
        # Empty DataFrame placeholder if no records
        df = pd.DataFrame()
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records for record_set\n  @id: {record_set_id}\n  columns: {df.columns.tolist() if not df.empty else '[]'}\n")

# For demonstration, use the first non-empty record set DataFrame
main_record_set_id = None
for rsid, df in dataframes.items():
    if not df.empty:
        main_record_set_id = rsid
        break
if main_record_set_id:
    print(f"Columns in '{main_record_set_id}':")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print("No record sets contain data.")

## 4. Exploratory Data Analysis (EDA)

We'll demonstrate numeric filtering, normalization, and grouping on the primary record set. Select a numeric field (by `@id`) from the record set's fields output above. If available fields are non-numeric, you may adjust this logic as needed for categoricals or booleans.


In [ ]:
# Identify a numeric field from the selected record set (by @id)
numeric_field_id = None
group_field_id = None

if main_record_set_id:
    fields = [f for rs in record_sets if rs.id == main_record_set_id for f in rs.fields]
    for field in fields:
        # Look for float or integer data type
        if field.data_type in ("Float", "Integer", "Number"):
            numeric_field_id = field.id
            break
    # For grouping, pick the first non-numeric (likely categorical) field
    for field in fields:
        if field.data_type in ("Text", "Boolean"):
            group_field_id = field.id
            break
    if numeric_field_id:
        df_main = dataframes[main_record_set_id]

        print(f"Using numeric field '@id': {numeric_field_id}")

        # Ensure numeric conversion
        df_main[numeric_field_id] = pd.to_numeric(df_main[numeric_field_id], errors="coerce")

        # Optionally drop NA in numeric field
        filtered_df = df_main[df_main[numeric_field_id].notna()]

        # Filter values above a threshold
        threshold = np.percentile(filtered_df[numeric_field_id], 50) # use median as example
        filtered_df = filtered_df[filtered_df[numeric_field_id] > threshold]

        print(f"Filtered records with {numeric_field_id} > {threshold} (median):\n")
        display(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
            filtered_df[numeric_field_id].std()
        )
        print(f"Normalized '{numeric_field_id}' for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by group_field
        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"Grouped data by '{group_field_id}':")
            display(grouped_df.head())

    else:
        print("No numeric fields detected for this record set.")
else:
    print("No main record set loaded for EDA.")

## 5. Visualization

Visualize the distribution of the selected numeric field, and, if appropriate, compare by group.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and numeric_field_id:
    df_main = dataframes[main_record_set_id]
    df_main[numeric_field_id] = pd.to_numeric(df_main[numeric_field_id], errors='coerce')
    plt.figure(figsize=(8, 4))
    sns.histplot(df_main[numeric_field_id].dropna(), kde=True, bins=20)
    plt.title(f"Distribution of '{numeric_field_id}'")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()
    
    if group_field_id and group_field_id in df_main.columns:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=df_main[group_field_id], y=df_main[numeric_field_id])
        plt.xticks(rotation=45)
        plt.title(f"Boxplot of '{numeric_field_id}' by '{group_field_id}'")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("No suitable numeric field available for visualization.")

## 6. Conclusion

In this notebook, you have explored the FAIR^2 dataset using the `mlcroissant` library, leveraging Croissant schema entity references by their `@id`. You learned how to:

- Load and inspect Croissant dataset metadata and structure
- Enumerate record sets and fields by `@id`
- Extract and preview records in pandas DataFrames based on `@id`
- Conduct basic EDA: filtering, normalization, and grouping using only `@id`-safe fields
- Visualize numeric distributions and group comparisons

You can now extend this approach for more advanced modeling and domain-specific analysis, always using Croissant-compliant references for robust, machine-actionable research.